In [0]:
select * from wiki_poc.poc.silver_recentchange_enwiki limit 1;

In [0]:
/*
Validation queries — run in a Databricks SQL cell after ≥ 5 min of pipeline

1. Volume sanity check (Silver should be ~5–15 % of Bronze)
*/
SELECT
  (SELECT COUNT(*) FROM wiki_poc.poc.silver_recentchange_enwiki)     AS silver_rows,
  (SELECT COUNT(*) FROM wiki_poc.poc.bronze_recentchange_raw)        AS bronze_rows,
  (SELECT COUNT(*) FROM wiki_poc.poc.silver_recentchange_quarantine) AS quarantine_rows;

In [0]:
-- 2. Quarantine breakdown by reason

SELECT quarantine_reason, COUNT(*) AS cnt,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct
FROM wiki_poc.poc.silver_recentchange_quarantine
GROUP BY quarantine_reason ORDER BY cnt DESC;

In [0]:
-- 3. Filter-correctness check (should return exactly one row: false | enwiki)
-- ──────────────────────────────────────────────────────────────────────────

SELECT DISTINCT bot, wiki FROM wiki_poc.poc.silver_recentchange_enwiki;

In [0]:
-- 4. Dedup check (should return 0 rows)
-- ──────────────────────────────────────

SELECT event_id, COUNT(*) AS dupes
FROM wiki_poc.poc.silver_recentchange_enwiki
GROUP BY event_id HAVING dupes > 1 LIMIT 10;

In [0]:
-- 5. byte_delta distribution
-- ────────────────────────────
SELECT
  MIN(byte_delta)              AS min_delta,
  MAX(byte_delta)              AS max_delta,
  ROUND(AVG(byte_delta), 1)    AS avg_delta,
  PERCENTILE(byte_delta, 0.5)  AS p50_delta,
  PERCENTILE(byte_delta, 0.95) AS p95_delta,
  COUNT_IF(byte_delta IS NULL) AS null_delta_rows   -- expected for new-page events
FROM wiki_poc.poc.silver_recentchange_enwiki;

In [0]:
-- 6. Top 10 pages by edit count in the last hour
-- ──────────────────────────────────────────────
SELECT title, COUNT(*) AS edits, COUNT(DISTINCT user) AS unique_editors
FROM wiki_poc.poc.silver_recentchange_enwiki
WHERE event_timestamp >= NOW() - INTERVAL 1 HOUR
GROUP BY title ORDER BY edits DESC LIMIT 10;

In [0]:
-- 7. DLT expectation failures (run as a DLT event log query)
-- ─────────────────────────────────────────────────────────
SELECT details:flow_name, details:expectations
FROM event_log('c40ae8e9-8371-477f-924b-13ec0af404f8')
WHERE event_type = 'flow_progress'
  AND details:flow_progress:metrics:num_rows_dropped_by_expectations > 0
ORDER BY timestamp DESC LIMIT 20;


In [0]:
-- Filter correctness — Silver contains only enwiki non-bot edits
SELECT DISTINCT bot, wiki, event_type
FROM wiki_poc.poc.silver_recentchange_enwiki;

In [0]:
-- Re-aggregate Silver for the last complete hour
WITH silver_agg AS (
  SELECT
    title,
    namespace,
    from_unixtime(FLOOR(unix_timestamp(event_timestamp) / 300) * 300) AS window_start,
    COUNT(*)              AS edit_count,
    SUM(byte_delta)       AS total_byte_delta
  FROM wiki_poc.poc.silver_recentchange_enwiki
  WHERE event_timestamp >= NOW() - INTERVAL 2 HOURS
    AND event_timestamp <  NOW() - INTERVAL 15 MINUTES  -- allow for watermark lag
  GROUP BY 1, 2, 3
),
gold AS (
  SELECT title, namespace, window_start, edit_count, total_byte_delta
  FROM wiki_poc.poc.gold_page_edits_5min
  WHERE window_start >= NOW() - INTERVAL 2 HOURS
    AND window_start <  NOW() - INTERVAL 15 MINUTES
)
SELECT
  COUNT(*)                                              AS windows_compared,
  SUM(ABS(s.edit_count - g.edit_count))                AS total_edit_count_diff,
  SUM(ABS(COALESCE(s.total_byte_delta,0) - COALESCE(g.total_byte_delta,0))) AS total_byte_diff,
  MAX(ABS(s.edit_count - g.edit_count))                AS max_edit_count_diff
FROM silver_agg s
JOIN gold g USING (title, namespace, window_start);

In [0]:
-- Bronze event_id coverage — no nulls
SELECT
  COUNT(*)                        AS total_rows,
  COUNT_IF(event_id IS NULL)      AS null_event_ids,
  COUNT_IF(raw_json IS NULL)      AS null_raw_json
FROM wiki_poc.poc.bronze_recentchange_raw;